**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Recurrent Neural Networks

The [Kalman workshop](./Intro_AdFilt_KF.ipynb) tracked a hidden state with a *hand-written* linear model. An RNN keeps the same architecture of ideas — hidden state in, observation out, state carried forward — but **learns the dynamics from data**, nonlinearity included. We build one in PyTorch and race it against the classical baselines on a noisy oscillator.

## 0. Introduction

Feedforward networks ([ANN workshop](../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb)) see each input in isolation. Sequences need *memory*: the RNN adds a loop —

$$\mathbf{h}_t = \tanh(W_{xh}\, \mathbf{x}_t + W_{hh}\, \mathbf{h}_{t-1} + \mathbf{b}) \qquad \mathbf{y}_t = W_{hy}\, \mathbf{h}_t$$

Compare the Kalman filter's $\hat{\mathbf{x}}_k = F\hat{\mathbf{x}}_{k-1} + K_k(\cdot)$: same skeleton, but $F$, $K$ are now *learned* and wrapped in a nonlinearity.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — tensors, `nn.Module`, the training loop.
- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — the state-space worldview.
- Install: `pip install torch` (CPU is fine for this workshop).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
print(torch.__version__)

2.13.0+cpu


---
### 🕐 Session 1 of 2 — *RNNs as Learned State-Space Models* (~35 min)
**Goal:** understand recurrence, backprop through time, vanishing gradients, and the LSTM fix.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb); [ANN](../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (training on a real sequence).

---

## 2. Theory

### 2.1. Unrolling & Backprop Through Time

💡 **Intuition.** To train an RNN, *unroll* the loop: an RNN run for 50 steps is a 50-layer feedforward network **whose layers all share the same weights**. Backprop works as usual on the unrolled graph ("backprop through time"); the only twist is that each weight receives blame from *every* time step it participated in.

### 2.2. Vanishing & Exploding Gradients

💡 **Intuition.** Blame flowing back through $T$ steps gets multiplied by (roughly) $W_{hh}$'s gain $T$ times. Gain a hair below 1 ⇒ the signal *vanishes* exponentially — the network can't learn long dependencies. A hair above 1 ⇒ it *explodes*. This is the same geometric-series dichotomy as $\sum x^n$ in [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb), and the same stability boundary as an IIR pole crossing the unit circle in [Filter Design](../Intro_DSP/Filter_Design.ipynb).

In [2]:
# Watch it happen: norm of d(h_T)/d(h_0) through a toy linear recurrence h_t = W h_{t-1}
for gain, label in [(0.9, "gain 0.9 → vanishes"), (1.1, "gain 1.1 → explodes")]:
    W = gain * torch.eye(8)
    g = torch.eye(8)
    norms = []
    for t in range(60):
        g = W.T @ g
        norms.append(g.norm().item())
    plt.semilogy(norms, label=label)
plt.legend(); plt.grid(True)
plt.xlabel("steps back in time"); plt.ylabel("gradient norm (log)")
plt.title("Why plain RNNs forget (or blow up)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/3553919686.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 2.3. LSTM: a Gated Memory Cell

The LSTM routes memory through an additive *cell state* $\mathbf{c}_t$ guarded by three learned gates — forget ($f$), input ($i$), output ($o$):

$$\mathbf{c}_t = f_t \odot \mathbf{c}_{t-1} + i_t \odot \tilde{\mathbf{c}}_t$$

Because $\mathbf{c}_t$ is updated by **addition** rather than repeated matrix multiplication, gradients can flow back along it without the exponential gain — a highway with learned on/off-ramps. (GRUs are the two-gate budget version; same spirit.)

---
### 🕐 Session 2 of 2 — *Sequence Prediction in Practice* (~40 min)
**Goal:** train an LSTM to forecast a noisy nonlinear oscillation; compare against classical baselines.
**Builds on:** Session 1.

---

## 3. Application: Forecasting a Noisy Oscillator

The target: a frequency-wobbling sinusoid — nonlinear enough that a fixed linear model struggles, structured enough to be learnable.

In [3]:
def make_signal(T=3000):
    t = np.arange(T) * 0.01
    inst_freq = 2.0 + 0.8 * np.sin(2 * np.pi * 0.05 * t)      # slowly wobbling frequency
    phase = 2 * np.pi * np.cumsum(inst_freq) * 0.01
    return (np.sin(phase) + 0.1 * rng.standard_normal(T)).astype(np.float32)

sig = make_signal()
split = 2400
train_sig, test_sig = sig[:split], sig[split:]

plt.figure(figsize=(9, 2.2))
plt.plot(sig[:800])
plt.title("The oscillator: frequency drifts, noise everywhere")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/148680797.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 3.1. Windowing the Sequence

Supervised framing: given the last `L` samples, predict the next one. (The same tapped-delay-line trick as the adaptive filters in the [APA workshop](./Intro_AdFilt_APA.ipynb) — the input representation is identical, only the model changes.)

In [4]:
L = 40

def windows(x, L):
    X = np.lib.stride_tricks.sliding_window_view(x, L)[:-1]    # (N, L)
    y = x[L:]                                                  # next sample
    return torch.from_numpy(X.copy()).unsqueeze(-1), torch.from_numpy(y.copy())

Xtr, ytr = windows(train_sig, L)
Xte, yte = windows(test_sig, L)
print("train windows:", tuple(Xtr.shape), " test windows:", tuple(Xte.shape))

train windows: (2360, 40, 1)  test windows: (560, 40, 1)


In [5]:
class Forecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                 # x: (batch, L, 1)
        out, _ = self.lstm(x)             # out: (batch, L, hidden)
        return self.head(out[:, -1]).squeeze(-1)   # predict from the LAST hidden state

model = Forecaster()
print(sum(p.numel() for p in model.parameters()), "parameters")

4513 parameters


In [6]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

model.train()
for epoch in range(8):
    total = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # the exploding-gradient seatbelt
        opt.step()
        total += loss.item() * len(xb)
    print(f"epoch {epoch}: train MSE {total / len(Xtr):.5f}")

epoch 0: train MSE 0.42116
epoch 1: train MSE 0.28398


epoch 2: train MSE 0.16730
epoch 3: train MSE 0.06473
epoch 4: train MSE 0.02403


epoch 5: train MSE 0.02086
epoch 6: train MSE 0.01896


epoch 7: train MSE 0.01820


### 3.2. The Bake-Off

Baselines every forecasting paper should be forced to include:

1. **Persistence** — "tomorrow = today." Embarrassingly strong on smooth signals.
2. **Linear AR(L)** — least-squares fit on the same windows: exactly the *Wiener* solution from the [APA workshop](./Intro_AdFilt_APA.ipynb), fitted in batch.

In [7]:
model.eval()
with torch.no_grad():
    pred_lstm = model(Xte).numpy()

pred_persist = Xte[:, -1, 0].numpy()

# Linear AR via least squares on the training windows
A = Xtr.squeeze(-1).numpy(); b = ytr.numpy()
w_ar, *_ = np.linalg.lstsq(A, b, rcond=None)
pred_ar = Xte.squeeze(-1).numpy() @ w_ar

truth = yte.numpy()
for name, pred in [("persistence", pred_persist), ("linear AR", pred_ar), ("LSTM", pred_lstm)]:
    print(f"{name:12s} test MSE: {np.mean((pred - truth)**2):.5f}")

persistence  test MSE: 0.03177
linear AR    test MSE: 0.01421
LSTM         test MSE: 0.01776


In [8]:
plt.figure(figsize=(9, 2.8))
plt.plot(truth[:300], "k", linewidth=1, label="truth")
plt.plot(pred_ar[:300], label="linear AR", alpha=0.8)
plt.plot(pred_lstm[:300], label="LSTM", alpha=0.8)
plt.legend(); plt.title("One-step-ahead forecasts on unseen data")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1845167/21433818.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Read that table carefully: **the linear AR model wins** — and that is the most important lesson in this workshop. One-step-ahead prediction of a smooth oscillation is a *nearly linear* task, and 40 least-squares coefficients beat 4k+ trained parameters. Deep models earn their keep only when the task outgrows linearity: try deepening the frequency wobble, raising the noise, or predicting 20 steps ahead instead of 1 — and watch the ranking flip. Never publish a neural forecaster without this bake-off.

## 4. Conclusion

An RNN is a learned nonlinear state-space model: unroll it to train it, gate it to remember, clip it to keep it stable. And always race it against persistence and a linear model before celebrating.

---
## Where next

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — replace recurrence with attention: all time steps talk directly, no vanishing highway needed.
- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — when you *know* the dynamics, the hand-derived gain is still king.
- [Scaling Neural Networks](../Intro_Mach_Learn/README.md#workshop-3--scaling-neural-networks-available) — what happens when models like these grow 1000×.